# JAX-CrossCat MNIST Benchmark (Colab)

Reproduces Section 3.2 of Mansinghka et al. (2016):
- Z-matrix of pixel dependence probabilities (Figure 13b)
- Spatial dependence map: foreground vs background pixels (Figure 13c)
- Inpainting with sparse observations (Figure 14)
- Digit classification with ROC curves (Figure 15)

**Runtime**: ~25-35 min on T4 GPU

| Section | Est. time |
|---------|----------|
| Setup | 2 min |
| Gibbs inference (4 chains × 500 sweeps) | 15-20 min |
| Inference queries (Z-matrix, classification, inpainting) | 5-10 min |
| Export | instant |

In [1]:
# Cell 1: GPU check
!nvidia-smi

: 

: 

In [2]:
# Cell 2: Install jaxcross (preserves Colab JAX+CUDA)
import os

WORKDIR = "/content/jaxcross"
BRANCH = "feat/arxiv-paper"  # @param {type:"string"}

!git clone https://github.com/sambhal-labs/jaxcross.git {WORKDIR} 2>/dev/null \
    || (cd {WORKDIR} && git pull)
os.chdir(WORKDIR)

!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

# --no-deps preserves Colab's pre-installed JAX+CUDA
%pip install -e . --no-deps -q
%pip install matplotlib scikit-learn -q

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for jax-crosscat (pyproject.toml) ... done


In [3]:
# Cell 3: Imports + verify GPU
import gc
import json
import time
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import softmax
from sklearn.datasets import fetch_openml
from sklearn.metrics import auc, confusion_matrix, roc_curve
from sklearn.svm import SVC

import crosscat
from crosscat.diagnostics import collect_diagnostics
from crosscat.model import initialize, log_joint
from crosscat.packed import pack_state
from crosscat.packed.kernels import packed_gibbs_sweep
from crosscat.packed.state import unpack_state
from crosscat.packed_inference import (
    batch_classify_column,
    batch_score_columns_binary,
    packed_dependence_matrix,
)
from crosscat.serialization import load_latest_checkpoint, load_state, save_checkpoint, save_state
from crosscat.types import ColumnType

RESULTS_DIR = Path("benchmarks/results/mnist")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = RESULTS_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"jaxcross version: {crosscat.__version__}")
print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
print(f"Backend: {jax.default_backend()}")
assert jax.default_backend() == "gpu", "GPU not available!"

jaxcross version: 0.10.1
JAX version: 0.7.2
Devices: [CudaDevice(id=0)]
Backend: gpu


---
## 1. Configuration

In [4]:
# ---- Benchmark Configuration ----
# Based on original CrossCat paper (Section 3.2)
PIXEL_SIZE = 16  # Downsample to 16x16
N_PIXELS = PIXEL_SIZE**2  # 256 binary pixels
N_SAMPLES = 1000  # 100 per digit (stratified)
N_CHAINS = 5
N_SWEEPS = 800
SEED = 42
DIAG_INTERVAL = 100  # Report diagnostics every N sweeps
CKPT_INTERVAL = 200  # Checkpoint every N sweeps

print(
    f"Config: {PIXEL_SIZE}x{PIXEL_SIZE} pixels ({N_PIXELS} binary + 1 categorical = {N_PIXELS + 1} cols)"
)
print(f"  {N_SAMPLES} samples (stratified), {N_CHAINS} chains x {N_SWEEPS} sweeps")

Config: 16x16 pixels (256 binary + 1 categorical = 257 cols)
  1000 samples (stratified), 5 chains x 800 sweeps


---
## 2. Data Loading

In [5]:
# Fetch MNIST with stratified sampling
print("Fetching MNIST...")
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
images = mnist.data  # (70000, 784)
labels = mnist.target.astype(int)  # (70000,)

# Stratified sample: 100 per digit
rng = np.random.default_rng(SEED)
selected = []
per_digit = N_SAMPLES // 10
for digit in range(10):
    idx = np.where(labels == digit)[0]
    chosen = rng.choice(idx, size=per_digit, replace=False)
    selected.extend(chosen)
selected = np.array(sorted(selected))

images_sel = images[selected]  # (1000, 784)
labels_sel = labels[selected]  # (1000,)

# Downsample 28x28 -> 16x16 and binarize
try:
    from skimage.transform import resize as sk_resize

    images_16 = np.array(
        [
            sk_resize(img.reshape(28, 28), (PIXEL_SIZE, PIXEL_SIZE), anti_aliasing=True).ravel()
            for img in images_sel
        ]
    )
except ImportError:
    from scipy.ndimage import zoom

    images_16 = np.array(
        [zoom(img.reshape(28, 28), PIXEL_SIZE / 28, order=1).ravel() for img in images_sel]
    )

# Binarize at threshold
images_bin = (images_16 / images_16.max() > 0.3).astype(np.float32)

# Build data matrix: 256 binary pixels + 1 categorical digit label
data_np = np.column_stack([images_bin, labels_sel.astype(np.float32)])
data_jax = jnp.array(data_np)

col_types = [ColumnType.BINARY] * N_PIXELS + [ColumnType.CATEGORICAL]
n_rows, n_cols = data_jax.shape
label_col = N_PIXELS  # index of digit label column

print(f"Data: {n_rows} rows x {n_cols} cols")
print(f"  Pixels: {N_PIXELS} binary, Label: 1 categorical (10 classes)")
print(f"  Digit distribution: {np.bincount(labels_sel)}")

Fetching MNIST...
Data: 1000 rows x 257 cols
  Pixels: 256 binary, Label: 1 categorical (10 classes)
  Digit distribution: [100 100 100 100 100 100 100 100 100 100]


---
## 3. Gibbs Inference (Packed, Batched Sweeps, Checkpointed)

Follows the WDI notebook pattern: batched `packed_gibbs_sweep`, checkpointing, `del` + `gc.collect()`.
If session dies, re-run this cell — it resumes from last checkpoint.

In [6]:
rng_key = jax.random.key(SEED)
init_keys = jax.random.split(rng_key, N_CHAINS)

states = []
all_chain_metrics = []
t_total = time.time()

for chain_idx in range(N_CHAINS):
    chain_ckpt_dir = CKPT_DIR / f"chain_{chain_idx + 1}"
    chain_result_path = RESULTS_DIR / f"chain_{chain_idx + 1}"

    # Try to load completed chain first (resume support)
    if chain_result_path.with_suffix(".jxc").exists():
        print(f"\n--- Chain {chain_idx + 1}/{N_CHAINS} --- LOADED from {chain_result_path}.jxc")
        state = load_state(chain_result_path, data=data_jax)
        states.append(state)
        all_chain_metrics.append([])
        continue

    print(f"\n--- Chain {chain_idx + 1}/{N_CHAINS} ---")
    k_i, k_sweep = jax.random.split(init_keys[chain_idx])

    # Try to resume from checkpoint
    start_sweep = 0
    try:
        packed, _, start_sweep = load_latest_checkpoint(chain_ckpt_dir)
        print(f"  Resuming from checkpoint at sweep {start_sweep}")
        for _ in range(start_sweep):
            k_sweep, _ = jax.random.split(k_sweep)
    except FileNotFoundError:
        state = initialize(k_i, data_jax, col_types)
        packed = pack_state(state)
        print(f"  Initialized: {state.n_views} views")
        del state  # free CPU memory

    chain_metrics = []
    t0 = time.time()

    # Run in batches of DIAG_INTERVAL sweeps for max GPU throughput
    sweep = start_sweep
    while sweep < N_SWEEPS:
        batch = min(DIAG_INTERVAL, N_SWEEPS - sweep)
        k_sweep, subkey = jax.random.split(k_sweep)
        packed = packed_gibbs_sweep(subkey, packed, data_jax, n_sweeps=batch)
        sweep += batch

        # Diagnostics — unpack temporarily, extract metrics, then free
        state_tmp = unpack_state(packed, col_types, data=data_jax)
        diag = collect_diagnostics(state_tmp, data_jax)
        n_views = state_tmp.n_views
        chain_metrics.append({"sweep": sweep, **diag})
        del state_tmp

        elapsed = time.time() - t0
        print(
            f"  Sweep {sweep:4d}/{N_SWEEPS}: views={n_views}, "
            f"log_joint={diag['log_joint']:.0f}, elapsed={elapsed:.0f}s"
        )

        # Checkpoint
        if sweep % CKPT_INTERVAL == 0:
            save_checkpoint(
                packed,
                chain_ckpt_dir,
                sweep,
                column_types=col_types,
                log_joint_value=float(diag["log_joint"]),
            )

    # Final unpack
    state = unpack_state(packed, col_types, data=data_jax)
    elapsed = time.time() - t0
    print(f"  Done: {elapsed:.1f}s ({elapsed / max(N_SWEEPS - start_sweep, 1):.2f}s/sweep)")

    # Save completed chain
    save_state(state, chain_result_path)
    print(f"  Saved to {chain_result_path}.jxc")

    states.append(state)
    all_chain_metrics.append(chain_metrics)
    del packed
    gc.collect()

print(f"\nTotal inference time: {time.time() - t_total:.1f}s")

# Build packed states for fast packed inference queries
packed_states = [pack_state(s) for s in states]
print(f"Packed {len(packed_states)} states for inference")


--- Chain 1/5 ---
  Initialized: 3 views


XlaRuntimeError: INTERNAL: CpuCallback error calling callback: Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
  File "/usr/local/lib/python3.12/dist-packages/tornado/platform/asyncio.py", line 211, in start
  File "/usr/lib/python3.12/asyncio/base_events.py", line 645, in run_forever
  File "/usr/lib/python3.12/asyncio/base_events.py", line 1999, in _run_once
  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py", line 510, in dispatch_queue
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py", line 

In [ ]:
# Convergence plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for c_idx, metrics in enumerate(all_chain_metrics):
    if not metrics:
        continue
    sweeps = [m["sweep"] for m in metrics]
    log_joints = [m["log_joint"] for m in metrics]
    n_views_list = [m.get("n_views", 0) for m in metrics]
    ax1.plot(sweeps, log_joints, "o-", label=f"Chain {c_idx + 1}")
    ax2.plot(sweeps, n_views_list, "o-", label=f"Chain {c_idx + 1}")

ax1.set_xlabel("Gibbs sweep")
ax1.set_ylabel("Log joint")
ax1.set_title("Convergence: Log Joint")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel("Gibbs sweep")
ax2.set_ylabel("Number of views")
ax2.set_title("Convergence: Views")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "convergence.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4. Z-Matrix and Pixel Dependence Map (Figures 13b, 13c)

In [ ]:
# Compute Z-matrix across all chains (packed, vectorized)
print("Computing Z-matrix...")
t0 = time.time()
z_matrix = np.array(packed_dependence_matrix(packed_states))
print(f"Z-matrix: {z_matrix.shape}, computed in {time.time() - t0:.1f}s")

# Figure 13b: Z-matrix heatmap
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(z_matrix, cmap="Greens", vmin=0, vmax=1, aspect="equal")
plt.colorbar(im, ax=ax, label="P(dependent)", shrink=0.8)
ax.set_title(f"MNIST Z-matrix ({n_cols} features)", fontsize=13)
ax.set_xlabel("Feature index")
ax.set_ylabel("Feature index")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "z_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# Figure 13c: Spatial dependence map
dep_with_label = z_matrix[label_col, :N_PIXELS]
dep_map = dep_with_label.reshape(PIXEL_SIZE, PIXEL_SIZE)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Continuous heatmap
im = axes[0].imshow(dep_map, cmap="RdYlBu_r", vmin=0, vmax=1)
plt.colorbar(im, ax=axes[0], label="P(dependent on digit)")
axes[0].set_title("Pixel dependence on digit label", fontsize=12)

# Binary: foreground vs background
threshold = 0.5
fg_mask = dep_map >= threshold
rgb = np.zeros((PIXEL_SIZE, PIXEL_SIZE, 3))
rgb[fg_mask] = [0.13, 0.59, 0.95]  # Blue = dependent
rgb[~fg_mask] = [0.91, 0.12, 0.39]  # Magenta = independent
axes[1].imshow(rgb)
axes[1].set_title(f"Foreground (blue) vs Background (magenta)\nThreshold={threshold}", fontsize=12)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.savefig(RESULTS_DIR / "pixel_dependence_map.png", dpi=150, bbox_inches="tight")
plt.show()

n_fg = fg_mask.sum()
print(f"Foreground pixels (dep >= {threshold}): {n_fg}/{N_PIXELS}")
print(f"Background pixels: {N_PIXELS - n_fg}/{N_PIXELS}")

---
## 5. Digit Classification + ROC Curves (Figure 15)

Uses `batch_classify_column` — vmapped over all rows and candidate values in one GPU call.

In [ ]:
# Select best chain (highest log_joint)
best_chain = max(
    range(len(states)),
    key=lambda i: float(
        all_chain_metrics[i][-1]["log_joint"]
        if all_chain_metrics[i]
        else log_joint(states[i], data_jax)
    ),
)
best_packed = packed_states[best_chain]
print(f"Best chain: {best_chain + 1} (views={states[best_chain].n_views})")

# 80/20 train/test split
rng_split = np.random.default_rng(SEED + 100)
n_test = n_rows // 5  # 200 test samples
test_idx = rng_split.choice(n_rows, size=n_test, replace=False)
test_idx.sort()
train_idx = np.setdiff1d(np.arange(n_rows), test_idx)

# Mask test labels (set to NaN)
data_test = data_np.copy()
data_test[test_idx, label_col] = np.nan
data_test_jax = jnp.array(data_test.astype(np.float32))

# Batch classify all test rows — average across all chains
print(f"Classifying {n_test} test samples (batch_classify_column, {N_CHAINS} chains)...")
t0 = time.time()
candidate_vals = jnp.arange(10, dtype=jnp.float32)
row_ids = jnp.array(test_idx)

all_logprobs = []
for pk in packed_states:
    logp = batch_classify_column(pk, data_test_jax, label_col, candidate_vals, row_ids)
    all_logprobs.append(np.array(logp))
avg_logprobs = np.mean(all_logprobs, axis=0)  # (n_test, 10)

preds = avg_logprobs.argmax(axis=1)
true_labels = labels_sel[test_idx]
accuracy = (preds == true_labels).mean()
print(f"Classification done in {time.time() - t0:.1f}s")
print(f"Accuracy: {accuracy:.1%} ({(preds == true_labels).sum()}/{n_test})")

cm = confusion_matrix(true_labels, preds, labels=range(10))
print("\nConfusion matrix:")
print(cm)

In [ ]:
# ROC curves: CrossCat vs SVM baselines
cc_probs = softmax(avg_logprobs, axis=1)

# SVM baselines
print("Training SVM baselines...")
X_train = images_bin[train_idx]
y_train = labels_sel[train_idx]
X_test = images_bin[test_idx]
y_test = labels_sel[test_idx]

svm_linear = SVC(kernel="linear", probability=True, random_state=SEED)
svm_linear.fit(X_train, y_train)
svm_linear_probs = svm_linear.predict_proba(X_test)
svm_linear_acc = svm_linear.score(X_test, y_test)
print(f"SVM linear accuracy: {svm_linear_acc:.1%}")

svm_rbf = SVC(kernel="rbf", probability=True, random_state=SEED)
svm_rbf.fit(X_train, y_train)
svm_rbf_probs = svm_rbf.predict_proba(X_test)
svm_rbf_acc = svm_rbf.score(X_test, y_test)
print(f"SVM RBF accuracy: {svm_rbf_acc:.1%}")

# Per-digit ROC
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.ravel()
cc_aucs, linear_aucs, rbf_aucs = [], [], []

for digit in range(10):
    y_bin = (y_test == digit).astype(int)
    fpr_cc, tpr_cc, _ = roc_curve(y_bin, cc_probs[:, digit])
    auc_cc = auc(fpr_cc, tpr_cc)
    cc_aucs.append(auc_cc)

    fpr_lin, tpr_lin, _ = roc_curve(y_bin, svm_linear_probs[:, digit])
    auc_lin = auc(fpr_lin, tpr_lin)
    linear_aucs.append(auc_lin)

    fpr_rbf, tpr_rbf, _ = roc_curve(y_bin, svm_rbf_probs[:, digit])
    auc_rbf = auc(fpr_rbf, tpr_rbf)
    rbf_aucs.append(auc_rbf)

    ax = axes[digit]
    ax.plot(fpr_cc, tpr_cc, "b-", linewidth=2, label=f"CrossCat ({auc_cc:.2f})")
    ax.plot(fpr_rbf, tpr_rbf, "r--", linewidth=1.5, label=f"SVM RBF ({auc_rbf:.2f})")
    ax.plot(fpr_lin, tpr_lin, "g:", linewidth=1.5, label=f"SVM Linear ({auc_lin:.2f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_title(f"Digit {digit}", fontsize=11)
    ax.set_xlabel("FPR", fontsize=9)
    ax.set_ylabel("TPR", fontsize=9)
    ax.legend(fontsize=7, loc="lower right")
    ax.grid(True, alpha=0.2)

fig.suptitle("Classification ROC Curves: CrossCat vs SVM", fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "classification_roc.png", dpi=150, bbox_inches="tight")
plt.show()

print(
    f"\nMean AUC — CrossCat: {np.mean(cc_aucs):.3f}, Linear: {np.mean(linear_aucs):.3f}, RBF: {np.mean(rbf_aucs):.3f}"
)

---
## 6. Pixel Inpainting (Figure 14)

Uses `batch_score_columns_binary` — vmapped over all missing pixels in one GPU call per row.

In [ ]:
# Inpainting at multiple observation fractions
obs_fractions = [0.10, 0.18, 0.30]
rng_inp = np.random.default_rng(SEED + 200)

# One image per digit for visualization
viz_rows = [np.where(labels_sel == d)[0][0] for d in range(10)]

inpaint_results = {}

for obs_frac in obs_fractions:
    n_obs = int(N_PIXELS * obs_frac)
    print(f"\n--- Inpainting at {obs_frac:.0%} ({n_obs}/{N_PIXELS} pixels) ---")

    accuracies = []
    for row_idx in viz_rows:
        obs_pixels = rng_inp.choice(N_PIXELS, size=n_obs, replace=False)
        missing_pixels = np.setdiff1d(np.arange(N_PIXELS), obs_pixels)

        # Mask unobserved pixels
        data_masked = data_np.copy()
        data_masked[row_idx, missing_pixels] = np.nan
        data_masked_jax = jnp.array(data_masked.astype(np.float32))

        # Score missing pixels — average across chains (one GPU call per chain)
        col_indices = jnp.array(missing_pixels)
        probs_sum = np.zeros(len(missing_pixels))
        for pk in packed_states:
            probs_sum += np.array(
                batch_score_columns_binary(pk, data_masked_jax, col_indices, row_idx)
            )
        probs_avg = probs_sum / len(packed_states)

        predicted = (probs_avg >= 0.5).astype(float)
        true_vals = data_np[row_idx, missing_pixels]
        accuracies.append(float((predicted == true_vals).mean()))

    mean_acc = np.mean(accuracies)
    inpaint_results[f"{obs_frac:.0%}"] = {
        "mean_accuracy": round(mean_acc, 4),
        "per_digit": [round(a, 4) for a in accuracies],
    }
    print(f"  Mean accuracy: {mean_acc:.1%}")
    print(f"  Per digit: {[f'{a:.1%}' for a in accuracies]}")

print("\nSummary:")
for frac, res in inpaint_results.items():
    print(f"  {frac} observed: {res['mean_accuracy']:.1%} accuracy")

In [ ]:
# Inpainting visualization (Figure 14 style)
obs_frac_viz = 0.30
n_obs_viz = int(N_PIXELS * obs_frac_viz)

fig, axes = plt.subplots(10, 4, figsize=(12, 25))
col_titles = ["Original", f"Observed ({obs_frac_viz:.0%})", "P(pixel=1)", "Predicted"]

rng_viz = np.random.default_rng(SEED + 300)  # fresh rng for viz

for d, row_idx in enumerate(viz_rows):
    obs_pixels = rng_viz.choice(N_PIXELS, size=n_obs_viz, replace=False)
    missing_pixels = np.setdiff1d(np.arange(N_PIXELS), obs_pixels)

    data_masked = data_np.copy()
    data_masked[row_idx, missing_pixels] = np.nan
    data_masked_jax = jnp.array(data_masked.astype(np.float32))

    # Score missing pixels
    col_indices = jnp.array(missing_pixels)
    probs_sum = np.zeros(len(missing_pixels))
    for pk in packed_states:
        probs_sum += np.array(
            batch_score_columns_binary(pk, data_masked_jax, col_indices, row_idx)
        )
    probs_avg = probs_sum / len(packed_states)

    # Build images
    prob_img = data_np[row_idx, :N_PIXELS].copy()
    prob_img[missing_pixels] = probs_avg

    obs_img = np.full((PIXEL_SIZE, PIXEL_SIZE, 3), [0.0, 0.8, 0.8])  # cyan = missing
    for p in obs_pixels:
        r, c = divmod(p, PIXEL_SIZE)
        val = data_np[row_idx, p]
        obs_img[r, c] = [val, val, val]

    pred_img = data_np[row_idx, :N_PIXELS].copy()
    pred_img[missing_pixels] = (probs_avg >= 0.5).astype(float)

    axes[d, 0].imshow(
        data_np[row_idx, :N_PIXELS].reshape(PIXEL_SIZE, PIXEL_SIZE), cmap="gray_r", vmin=0, vmax=1
    )
    axes[d, 1].imshow(obs_img)
    axes[d, 2].imshow(prob_img.reshape(PIXEL_SIZE, PIXEL_SIZE), cmap="gray_r", vmin=0, vmax=1)
    axes[d, 3].imshow(pred_img.reshape(PIXEL_SIZE, PIXEL_SIZE), cmap="gray_r", vmin=0, vmax=1)

    axes[d, 0].set_ylabel(f"Digit {d}", fontsize=11)
    for j in range(4):
        axes[d, j].set_xticks([])
        axes[d, j].set_yticks([])
        if d == 0:
            axes[d, j].set_title(col_titles[j], fontsize=11)

fig.suptitle(f"Pixel Inpainting at {obs_frac_viz:.0%} Observation", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "inpainting.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 7. Contingency Table

In [ ]:
# Digit-to-cluster contingency for best chain
best_state = states[best_chain]
print(f"Best chain: {best_chain + 1}, {best_state.n_views} views")
for v_idx, view in enumerate(best_state.views):
    n_clusters = int(np.array(view.row_assignments).max()) + 1
    n_cols_in_view = len(view.column_indices)
    print(f"  View {v_idx}: {n_cols_in_view} cols, {n_clusters} clusters")

# Find view containing digit label
label_view = None
for v_idx, view in enumerate(best_state.views):
    if label_col in view.column_indices:
        label_view = v_idx
        break

if label_view is not None:
    assignments = np.array(best_state.views[label_view].row_assignments)
    n_clusters = int(assignments.max()) + 1
    print(f"\nLabel in View {label_view} ({n_clusters} clusters)")

    contingency = np.zeros((10, n_clusters), dtype=int)
    for row in range(n_rows):
        contingency[int(labels_sel[row]), int(assignments[row])] += 1

    fig, ax = plt.subplots(figsize=(max(8, n_clusters * 0.6), 5))
    im = ax.imshow(contingency, cmap="Blues", aspect="auto")
    plt.colorbar(im, ax=ax, label="Count")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Digit")
    ax.set_yticks(range(10))
    ax.set_xticks(range(n_clusters))
    ax.set_title(f"Digit-Cluster Contingency (View {label_view})")
    for i in range(10):
        for j in range(n_clusters):
            if contingency[i, j] > 0:
                ax.text(j, i, str(contingency[i, j]), ha="center", va="center", fontsize=7)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "contingency.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Digit label is in its own view (independent of pixels)")

---
## 8. Export Results

In [ ]:
import subprocess

summary = {
    "config": {
        "pixel_size": PIXEL_SIZE,
        "n_pixels": N_PIXELS,
        "n_samples": N_SAMPLES,
        "n_chains": N_CHAINS,
        "n_sweeps": N_SWEEPS,
        "backend": jax.default_backend(),
        "device": str(jax.devices()[0]),
    },
    "structure": {
        f"chain_{i + 1}": {
            "n_views": states[i].n_views,
            "cols_per_view": [len(v.column_indices) for v in states[i].views],
        }
        for i in range(N_CHAINS)
    },
    "classification": {
        "crosscat_accuracy": round(float(accuracy), 4),
        "svm_linear_accuracy": round(float(svm_linear_acc), 4),
        "svm_rbf_accuracy": round(float(svm_rbf_acc), 4),
        "crosscat_mean_auc": round(float(np.mean(cc_aucs)), 4),
        "svm_linear_mean_auc": round(float(np.mean(linear_aucs)), 4),
        "svm_rbf_mean_auc": round(float(np.mean(rbf_aucs)), 4),
    },
    "inpainting": inpaint_results,
}

with open(RESULTS_DIR / "metrics.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 60)
print("MNIST BENCHMARK RESULTS")
print("=" * 60)
print(json.dumps(summary, indent=2))

# Archive
archive = "/content/mnist_benchmark_results.tar.gz"
subprocess.run(
    ["tar", "czf", archive, "-C", str(RESULTS_DIR.parent), RESULTS_DIR.name],
    check=True,
)
print(f"\nArchive: {archive}")
for f in sorted(RESULTS_DIR.glob("*")):
    if f.is_file():
        print(f"  {f.name} ({f.stat().st_size:,} bytes)")
print("\nDownload from Colab Files panel (left sidebar).")